### E-Commerce US Dataset -- Feature Engineering and validation

\Importing Necessary Libraries

In [4]:
import numpy as np
import pandas as pd
import os

import warnings
warnings.filterwarnings("ignore")

\Loading the processed data

In [5]:
path = r"C:\NG\E-Commerce US dataset\E-Commerce-US-dataset\data\processed"

master = pd.read_csv(os.path.join(path,"master_clean.csv"))
items = pd.read_csv(os.path.join(path,"items_clean.csv"))

In [6]:
master.shape

(10000, 39)

In [7]:
items.shape

(21838, 23)

\Date data type conversion

In [8]:
for col in ["order_purchase_timestamp","order_delivered_customer_date",
          "order_estimated_delivery_date"]:
    if col in master.columns:
        master[col] = pd.to_datetime(master[col],errors = "coerce")

In [9]:
items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"],errors="coerce")

### Feature Engineering

\Delivery Time Metrics

In [10]:
#delivery days
master["delivery_days"] = (master["order_delivered_customer_date"] - master["order_purchase_timestamp"]).dt.days

#estimated delivery days
master["estimated_days"] = (master["order_estimated_delivery_date"] - master["order_purchase_timestamp"]).dt.days

#delivery delay
master["delivery_delay"] = master["delivery_days"] - master["estimated_days"]

#late flag
master["is_late"] = (master["delivery_delay"]> 0).astype(int)

#Delivery Speed Category
master["delivery_speed"] = pd.cut(master["delivery_days"],bins=[0,5,10,100],labels=["Fast","Medium","Slow"])

\Average Order Value + Payment Behavior

In [11]:
# avg value per item
master["avg_item_value"] = (master["total_price"]/master["n_items"]).round(2)

#Freight ratio
master["freight_ratio"] = (master["total_freight"]/ (master["total_price"] + 0.01) *100).round(2)

#payment behavior
master["uses_installments"] = (master["max_installments"]>1).astype(int)
master["installment_category"] = pd.cut(master["max_installments"],bins=[0,1,6,100],
                                        labels=["Single","Short-term","Long-term"])

\Profit feature

In [12]:
items["profit"] = (items["price"] - items["cost"]).round(2)
items["profit_margin_pct"] = ((items["profit"] / items["price"]) * 100).round(2)

\Customer Life Metrics

In [13]:
#customer table
customer_features = master.groupby("customer_unique_id").agg(
    total_orders     = ("order_id", "count"),
    total_spend      = ("total_payment", "sum"),
    avg_order_value  = ("total_payment", "mean"),
    total_items      = ("n_items", "sum"),
    avg_review_given = ("avg_review_score", "mean"),
    first_order      = ("order_purchase_timestamp", "min"),
    last_order       = ("order_purchase_timestamp", "max"),
).reset_index()

# Repeat customer
customer_features["is_repeat_customer"] = (customer_features["total_orders"] > 1).astype(int)

# Customer tenure 
customer_features["tenure_days"] = (customer_features["last_order"] -
                                     customer_features["first_order"]).dt.days

# Customer value tier
customer_features["value_tier"] = pd.qcut(customer_features["total_spend"],
                                           q=4, labels=["Bronze","Silver","Gold","Platinum"])

\Customer Engagement

In [14]:
customer_features["review_rate"] = master.groupby("customer_unique_id")["has_review"].mean().values
orders_norm = customer_features["total_orders"] / customer_features["total_orders"].max()
spend_norm  = customer_features["total_spend"]  / customer_features["total_spend"].max()
customer_features["engagement_score"] = (
    orders_norm * 0.4 + customer_features["review_rate"] * 0.3 + spend_norm * 0.3
).round(3)

In [15]:
customer_features["engagement_score"].describe()

count    7898.000000
mean        0.393631
std         0.088765
min         0.067000
25%         0.374000
50%         0.389000
75%         0.420000
max         0.900000
Name: engagement_score, dtype: float64

\Seller Performance Metrics

In [16]:
#seller Table
seller_features = items.groupby("seller_id").agg(
    total_revenue = ("price","sum"),
    total_items_sold = ("order_id","count"),
    avg_price = ("price","mean"),
    n_unique_orders = ("order_id","nunique"),
    n_products = ("product_id","nunique")
).reset_index()

#revenue per order
seller_features["revenue_per_order"] = (seller_features["total_revenue"]/seller_features["n_unique_orders"]).round(2)

#seller tier
seller_features["seller_tier"] = pd.qcut(seller_features["total_revenue"],q=3,labels=["Low","Mid","Top"])


\Product Popularity Metrics

In [17]:
#product table
product_features = items.groupby("product_id").agg(
    times_ordered = ("order_id", "count"),
    total_revenue = ("price", "sum"),
    avg_price = ("price", "mean")
).reset_index()

#product popularity tier
product_features["popularity_tier"] = pd.qcut(product_features["times_ordered"],
                                               q=3, labels=["Low","Medium","High"],
                                               duplicates="drop")

#revenue contribution rank
product_features["product_revenue_rank"] = product_features["total_revenue"].rank(ascending=False).astype(int)

\Saving the processed + feature added files 

In [18]:
out_path = r"C:\NG\E-Commerce US dataset\E-Commerce-US-dataset\data\processed-feature"
master.to_csv(os.path.join(out_path, "master_features.csv"), index=False)
customer_features.to_csv(os.path.join(out_path, "customer_features.csv"), index=False)
seller_features.to_csv(os.path.join(out_path, "seller_features.csv"), index=False)
product_features.to_csv(os.path.join(out_path, "product_features.csv"), index=False)

### Data Validation

In [38]:
class DataValidator:

    def __init__(self,df,name="dataset"):
        self.df = df
        self.name = name
        self.results = []

    def _record(self,category,rule,passed,detail=""):
        self.results.append({
            "category" : category,
            "rule" : rule,
            "passed" : "Pass" if passed else "Fail",
            "detail" : detail 
        })


    #schema validation
    def validate_schema(self,expected_columns):
        actual = set(self.df.columns)
        missing_coln = set(expected_columns) - actual
        passed = len(missing_coln) ==0
        self._record("Schema","All Expected Columns Present",passed,f"Missing: {missing_coln}" if missing_coln else "OK" )
        return self

    #data type Validation
    def validate_dtypes(self,dtype_map):
        for col, expected in dtype_map.items():
            if col in self.df.columns:
                actual = str(self.df[col].dtypes)
                passed = expected in actual
                self._record("Data Type",f"{col} is {expected}",passed,f"actual: {actual}")
        return self

    #Missing Value Validation
    def validate_missing(self,max_missing_pct = 5,exclude=None):
        exclude = exclude or []
        for col in self.df.columns:
            if col in exclude:
                continue
            pct = self.df[col].isna().mean() *100
            passed = pct <= max_missing_pct
            if not passed:
                self._record("Missing",f"{col} <= {max_missing_pct}% Missing", passed, f"actual: {pct:.1f}%")

        if not any(r["category"]=="Missing" for r in self.results):
            self._record("Missing",f"All Columns <= {max_missing_pct}% Missing", True , "OK")
        return self

    #Business Rule Validation
    def validate_business_rule(self,rule_name,condition_series):
        passed = condition_series.all()
        n_fails = (~condition_series).sum()
        self._record("Business Rule",rule_name,passed, f"{n_fails} violations" if not passed else "Ok")
        return self

    #Referential Integrity
    def validate_reference(self,col,reference_value,ref_name):
        orphans = (~self.df[col].isin(reference_value)).sum()
        passed = orphans== 0
        self._record("Reference Integrity",f"{col} in {ref_name}",passed,f"{orphans} orphans" if orphans else "Ok")
        return self

    #Consistency Validation
    def validate_consistency(self,rule_name, condition_series):
        passed = condition_series.all()
        n_fails = (~condition_series).sum()
        self._record("Consistency Validation",rule_name,passed,f"{n_fails} inconsistent" if not passed else "Ok")
        return self

    def report(self):
        r = pd.DataFrame(self.results)
        n_pass = (r["passed"] == "Pass").sum()
        print(f"\n{'='*60}")
        print(f"  VALIDATION REPORT: {self.name}  ({n_pass}/{len(r)} passed)")
        print(f"{'='*60}")
        print(r.to_string(index=False))
        return r
    

\Validating the file

In [39]:
customers = pd.read_csv(r"C:\NG\E-Commerce US dataset\E-Commerce-US-dataset\data\processed-feature\customer_features.csv")

In [40]:
# validating master file
v = DataValidator(master,"master_features")

# schema 
v.validate_schema([
    "order_id", "customer_id", "order_status",
    "total_payment", "total_price", "customer_state"
])

# dtypes
v.validate_dtypes({
    "total_payment": "float",
    "n_items": "int",
    "customer_age": "int",
})

# Missing values 
v.validate_missing(max_missing_pct=10,
                   exclude=["order_delivered_customer_date","avg_review_score",
                            "delivery_days","is_late"])

# Business Rule
v.validate_business_rule("payment >=0", master["total_payment"].dropna() >=0)
v.validate_business_rule("age -18 to 100", master["customer_age"].between(18,100))
v.validate_business_rule("n_items >=0", master["n_items"] >=0)

#Referential Integrity
v.validate_reference("customer_unique_id",set(customers["customer_unique_id"]),"customer_table")

#consistency
city_name = master.groupby("customer_city")["customer_state"].nunique()
v.validate_consistency("Each city -> 1 state", city_name <= 1)


In [41]:
master_report = v.report()
master_report


  VALIDATION REPORT: master_features  (9/12 passed)
              category                                         rule passed          detail
                Schema                 All Expected Columns Present   Pass              OK
             Data Type                       total_payment is float   Pass actual: float64
             Data Type                               n_items is int   Pass   actual: int64
             Data Type                          customer_age is int   Pass   actual: int64
               Missing order_estimated_delivery_date <= 10% Missing   Fail   actual: 60.2%
               Missing                estimated_days <= 10% Missing   Fail   actual: 60.2%
               Missing                delivery_delay <= 10% Missing   Fail   actual: 63.4%
         Business Rule                                  payment >=0   Pass              Ok
         Business Rule                               age -18 to 100   Pass              Ok
         Business Rule               

,category,rule,passed,detail
0,Schema,All Expected Columns Present,Pass,OK
1,Data Type,total_payment is float,Pass,actual: float64
2,Data Type,n_items is int,Pass,actual: int64
3,Data Type,customer_age is int,Pass,actual: int64
4,Missing,order_estimated_delivery_date <= 10% Missing,Fail,actual: 60.2%
5,Missing,estimated_days <= 10% Missing,Fail,actual: 60.2%
6,Missing,delivery_delay <= 10% Missing,Fail,actual: 63.4%
7,Business Rule,payment >=0,Pass,Ok
8,Business Rule,age -18 to 100,Pass,Ok
9,Business Rule,n_items >=0,Pass,Ok


\Saving the validation report

In [42]:
VAL = r"C:\NG\E-Commerce US dataset\E-Commerce-US-dataset\reports\Validation_report"
master_report.to_csv(os.path.join(VAL, "master_validation.csv"), index=False)